In [1]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import os
import random
import gc
import sys
import pprint
import scrublet as scr

In [2]:
# --- Set a global random seed for reproducibility ---
seed = 97
random.seed(seed)
np.random.seed(seed)

print(f"Global random seed set to: {seed}")

# --- Configure plotting parameters ---
sc.set_figure_params(frameon=False, fontsize=10, dpi=100)
# This section demonstrates how to add a custom module path.
# The 'anno_self' module contains user-defined annotation functions.
# Ensure this path points to the directory containing your custom scripts.
try:
    sys.path.append(r"/home/lishaogang/NEU/validation-set")
    import anno_self
    print("Custom annotation module 'anno_self' loaded successfully.")
except ImportError:
    print("Warning: Custom annotation module 'anno_self' not found. Annotation steps will require manual implementation.")

In [3]:
os.chdir("/home/lishaogang/NEU2026-Jan")

In [4]:
adata = sc.read_h5ad('adata_0127.adata')

In [5]:
import scanpy as sc
import matplotlib.pyplot as plt
import matplotlib as mpl

# ⭐关键：SVG文字不转路径
mpl.rcParams['svg.fonttype'] = 'none'
mpl.rcParams['pdf.fonttype'] = 42

sc.set_figure_params(figsize=(8, 4))

ax = sc.pl.violin(
    adata,
    keys=['n_genes_by_counts', 'pct_counts_mt'],
    groupby='dataset',
    jitter=False,
    stripplot=False,
    multi_panel=True,
    rotation=45,
    show=False
)

# ⭐关键：只 rasterize violin
for a in plt.gcf().axes:
    for coll in a.collections:
        coll.set_rasterized(True)

plt.savefig("violin_final.svg", bbox_inches="tight")

In [6]:
import scanpy as sc
import matplotlib.pyplot as plt
import matplotlib as mpl

# ⭐关键：SVG文字不转路径
mpl.rcParams['svg.fonttype'] = 'none'
mpl.rcParams['pdf.fonttype'] = 42

sc.set_figure_params(figsize=(8, 4))

ax = sc.pl.violin(
    adata,
    keys=['n_genes_by_counts', 'pct_counts_mt'],
    groupby='dataset',
    jitter=False,
    stripplot=False,
    multi_panel=True,
    rotation=45,
    show=False
)

# ⭐关键：只 rasterize violin
for a in plt.gcf().axes:
    for coll in a.collections:
        coll.set_rasterized(True)

plt.savefig("violin_final.svg", bbox_inches="tight")

# ===================== 统计 n_genes_by_counts 最小值 =====================
min_n_genes = adata.obs['n_genes_by_counts'].min()
print(f"n_genes_by_counts 最小值: {min_n_genes}")

# 如果你想按 dataset 分组分别看最小值
min_by_dataset = adata.obs.groupby('dataset')['n_genes_by_counts'].min()
print("\n按 dataset 分组的 n_genes_by_counts 最小值:")
print(min_by_dataset)

In [7]:
# 查看总基因数
print("当前总基因数量:", adata.n_vars)

# 查看前20个基因的表达总和（判断是不是低表达）
print("\n各基因总表达量 (前20):")
print(adata.X.sum(axis=0).A1[:20])

In [5]:
adata_B = sc.read_h5ad('adata_B_final.h5ad')

In [6]:
adata_NK = sc.read_h5ad('adata_NK_final.h5ad')

In [7]:
adata_Myeloid = sc.read_h5ad('adata_Myeloid_final.h5ad')

In [ ]:
adata_T = sc.read_h5ad('adata_T_final.h5ad')

In [9]:
sc.pl.umap(adata,color = 'cell_subtype')

In [10]:
adata.obs['cell_subtype'] = 'cell'

In [33]:
import pandas as pd
import scanpy as sc

# 1. 确保cell_subtype列为字符串类型（先复制到临时变量，避免链式赋值）
cell_subtype_series = adata.obs['cell_subtype'].astype(str)

# 2. 提取所有子集的cell_subtype信息并合并
combined_subtypes = pd.concat([
    adata_B.obs['cell_subtype'],
    adata_NK.obs['cell_subtype'],
    adata_Myeloid.obs['cell_subtype'],
    adata_T.obs['cell_subtype']  # 如需合并T细胞子集可取消注释
])

# 3. 修复核心：先更新临时Series，再赋值回adata.obs（避免链式赋值）
cell_subtype_series.update(combined_subtypes)
adata.obs['cell_subtype'] = cell_subtype_series  # 直接赋值到原始对象

# 4. 转回分类类型
adata.obs['cell_subtype'] = adata.obs['cell_subtype'].astype('category')

# 5. 验证结果
print(adata.obs['cell_subtype'].value_counts())



In [34]:
import pandas as pd
import scanpy as sc

# 步骤1：筛选出cell_type为pDC的细胞
mask = adata.obs['cell_type'] == 'pDC'

# 步骤2：关键修复——先给cell_subtype添加新类别pDC（如果不存在）
if 'pDC' not in adata.obs['cell_subtype'].cat.categories:
    adata.obs['cell_subtype'] = adata.obs['cell_subtype'].cat.add_categories('pDC')

# 步骤3：再赋值（此时pDC已在类别中，不会报错）
adata.obs.loc[mask, 'cell_subtype'] = 'pDC'

# 验证修改结果
print(f"pDC细胞的cell_subtype分布：")
print(adata.obs[mask]['cell_subtype'].value_counts())

In [36]:
import scanpy as sc

# 1. 先查看待删除细胞的数量（验证筛选逻辑）
mask = adata.obs['cell_subtype'] == 'cell'
print(f"待删除的'cell'注释细胞数量：{mask.sum()}")

# 2. 反向筛选：保留非'cell'的细胞（核心操作）
adata = adata[~mask].copy()  # ~表示取反，copy()避免视图警告

# 3. 验证删除结果
print(f"删除后剩余细胞数量：{adata.n_obs}")
print(f"剩余细胞的cell_subtype唯一值：{adata.obs['cell_subtype'].unique()}")

In [37]:
# 6. 可视化
sc.pl.umap(adata, color='cell_subtype'
           #, legend_loc='on data'
          )


In [44]:
import scanpy as sc
import matplotlib.pyplot as plt

# 1. 手动创建画布和坐标轴，精准控制大小
fig, ax = plt.subplots(figsize=(6, 5)) 

# 2. 将创建好的 ax 传给 scanpy
sc.pl.umap(
    adata,
    color='cell_subtype',
    palette='tab20',
    size=3,
    ax=ax,                # 关键：指定在哪个坐标轴上画
    show=False,           # 关键：先不显示，方便后续调整
    frameon=False,
    legend_loc='right margin' # 建议用这个控制图例
)

# 3. 额外调小整体边距（主体占满画布）
# left, right, top, bottom 取值 0-1 之间
plt.subplots_adjust(left=0.05, right=0.85, top=0.95, bottom=0.05)

# 4. 最后显示
plt.show()

In [46]:
import scanpy as sc
import matplotlib.pyplot as plt

# 1. 手动创建画布和坐标轴，精准控制大小
fig, ax = plt.subplots(figsize=(6, 5)) 

# 2. 将创建好的 ax 传给 scanpy
sc.pl.umap(
    adata,
    color='dataset',
    palette='tab20',
    size=3,
    ax=ax,                # 关键：指定在哪个坐标轴上画
    show=False,           # 关键：先不显示，方便后续调整
    frameon=False,
    legend_loc='right margin' # 建议用这个控制图例
)

# 3. 额外调小整体边距（主体占满画布）
# left, right, top, bottom 取值 0-1 之间
plt.subplots_adjust(left=0.05, right=0.85, top=0.95, bottom=0.05)

# 4. 最后显示
plt.show()

In [53]:
adata.write_h5ad("adata_final0.h5ad")

In [13]:
adata = sc.read_h5ad("adata_final0.h5ad")

In [14]:
import plotly.graph_objects as go
import pandas as pd
import numpy as np

# 1. 提取 UMAP 坐标和元数据
df = pd.DataFrame(adata.obsm['X_umap'], columns=['UMAP1', 'UMAP2'])
df['cell_id'] = adata.obs_names.tolist()
df['cell_subtype'] = adata.obs['cell_subtype'].astype(str).values

# 为不同类别生成颜色（可选，方便圈选时区分）
# 简单的颜色映射
types = df['cell_subtype'].unique()
color_map = {t: i for i, t in enumerate(types)}
df['color_idx'] = df['cell_subtype'].map(color_map)

# 2. 创建空的 FigureWidget
fig = go.FigureWidget()



In [7]:
# 3. 添加散点数据 (关键：记录下这个 trace)
fig.add_scattergl(
    x=df['UMAP1'],
    y=df['UMAP2'],
    mode='markers',
    marker=dict(
        size=3, 
        color=df['color_idx'], 
        colorscale='Viridis', 
        opacity=0.6,
        showscale=False
    ),
    text=df['cell_subtype'],
    hoverinfo='text'
)

# 明确指定我们要绑定事件的散点对象
scatter_trace = fig.data[0]

# 用于全局存储选中的 ID
selected_cell_ids = []

# 4. 定义回调函数
def selection_fn(trace, points, selector):
    global selected_cell_ids
    # 获取选中点的 cell_id
    selected_cell_ids = [df.iloc[p]['cell_id'] for p in points.point_inds]
    print(f"✅ 捕获成功！已选中 {len(selected_cell_ids)} 个细胞。请去运行下一步的删除代码。")

# 5. 绑定选中事件到散点对象（trace）上
scatter_trace.on_selection(selection_fn)

# 6. 设置布局
fig.update_layout(
    dragmode='lasso', 
    width=900, 
    height=700, 
    title="【手动圈选模式】使用右上角套索工具圈选中间杂乱细胞",
    template='none'
)

print("操作提示：")
print("1. 点击图表右上角工具栏中的 'Lasso Select'（套索图标）。")
print("2. 在图中手动圈出你想删掉的那团点。")
print("3. 圈完后不要刷新页面，直接运行下方的删除代码单元格。")

fig

In [24]:
adata.obs.columns

In [15]:
# 1. 检查选中的 ID 数量
print(f"准备修改标签的细胞数量: {len(selected_cell_ids)}")

if len(selected_cell_ids) > 0:
    # 2. 处理 Categorical 类型（如果是 category 类型，必须先添加新类别）
    if hasattr(adata.obs['cell_subtype'], 'cat'):
        if 'cell' not in adata.obs['cell_subtype'].cat.categories:
            adata.obs['cell_subtype'] = adata.obs['cell_subtype'].cat.add_categories(['cell'])
    
    # 3. 将选中的细胞 ID 对应的标签改为 'cell'
    # .loc[索引列表, 列名] 是最安全且推荐的修改方式
    adata.obs.loc[selected_cell_ids, 'cell_subtype'] = 'cell'
    
    print("✅ 修改成功！")
    # 4. 查看一下修改后的统计
    print("\n修改后的细胞分布：")
    print(adata.obs['cell_subtype'].value_counts())
else:
    print("❌ 错误：未发现选中的细胞，请先在图上进行圈选。")

In [11]:
# 重新整理的基因列表，按预期的“阶梯”顺序排列
# 删除了可能引起报错的常用名，全部使用图像中出现的官方 Symbol
marker_genes = {
    'B cells': ['CD19', 'MS4A1', 'IGHD', 'TCL1A', 'FCER2', 'BACH2', 'SOX4'],
    'Plasma': ['CD27', 'SDC1', 'JCHAIN', 'XBP1', 'MZB1'],
    'Naive T': ['CD3D', 'TRAC', 'CCR7', 'SELL', 'LEF1', 'TCF7'],
    'CD4 T': ['CD4', 'GPR183', 'IL2RA', 'CTLA4'],
    'CD8 T': ['CD8A', 'CD8B'],
    'Cytotoxic': ['GZMA', 'GZMH', 'NKG7', 'GNLY', 'PRF1', 'FCGR3A'],
    'Innate T/NK': ['SLC4A10', 'KLRB1', 'TRDC', 'TRDV2', 'NCAM1', 'XCL1'],
    'Monocytes': ['CD14', 'S100A8', 'S100A9', 'VCAN', 'FCN1'],
    'Macrophages': ['C1QA', 'APOE', 'CD163', 'F13A1', 'LYVE1', 'FOLR2'],
    'DC/Neutrophil': ['FCGR3B', 'CSF3R', 'CLEC10A', 'HLA-DRA', 'AXL', 'SIGLEC6','CEACAM8']
}

In [12]:
# 按照从左上到右下的顺序排列细胞类型
cell_order = [
    # B系列 (对应左上)
    'Naive B cell', 'Transitional B cell', 'Memory B cell', 'Plasma cell',
    # T系列
    'CD4 Naïve', 'CD8 Naïve', 'CD4 Memory', 'CD8 Memory', 'CD4 Treg', 'CD4 CTL', 'CD8 CTL',
    # 固有淋巴系
    'MAIT', 'γδT', 'ILC2', 'NK bright', 'Transit. NK', 'NK dim', 'NKT',
    # 髓系 (对应右下)
    'Classical Monocytes', 'Non-classical Monocytes', 'Neutrophils',
    'IFN-γ-Responsive Macrophages', 'Complement-Activated Macrophages',
    'Phagocytic Inflammatory Macrophages', 'Inflammatory Activated Macrophages',
    'Metabolically Active Resting Macrophages', 'Tissue-Resident Macrophages',
    'cDC2', 'pDC', 'ASDC','cell'
]

# 执行排序
adata.obs['cell_subtype'] = adata.obs['cell_subtype'].astype('category')
actual_order = [c for c in cell_order if c in adata.obs['cell_subtype'].cat.categories]
adata.obs['cell_subtype'] = adata.obs['cell_subtype'].cat.reorder_categories(actual_order)

In [13]:
import scanpy as sc

sc.pl.dotplot(
    adata,
    marker_genes,
    groupby='cell_subtype',
    standard_scale='var',      # 核心：将每个基因的表达缩放到0-1，凸显阶梯感
    use_raw=True,              # 通常 marker 在 raw 空间更清晰
    cmap='Reds',
    figsize=(18, 10),          # 适当调整大小以容纳所有亚型
    var_group_rotation=0
    #,      # 顶部分组文字角度
   # x_label_rotation=90,       # 底部基因名角度
    # categories_order=actual_order, # 也可以在这里再次指定顺序
)

In [19]:
import scanpy as sc

# 1. 关键：添加 groups=['cell']，只计算这一组的差异
sc.tl.rank_genes_groups(
    adata, 
    groupby='cell_subtype', 
    groups=['cell'],      # <--- 极其重要：只算你要的那一组
    reference='rest',     # 剩下的作为对照
    method='t-test'
)



In [20]:
# 2. 提取 'cell' 组的前 10 个基因
marker_df = sc.get.rank_genes_groups_df(adata, group='cell')


In [21]:
top10_genes = marker_df.head(10)['names'].tolist()

print("‘cell’类型的 Top 10 Marker 基因：")
print(top10_genes)

# 3. 绘制气泡图
sc.pl.dotplot(
    adata, 
    var_names=top10_genes, 
    groupby='cell_subtype', 
    standard_scale='var',
    title="Top 10 markers for 'cell'"
)

In [30]:
sc.pl.umap(adata, color=['cell_subtype'], vmin=0)

In [31]:
sc.pl.umap(adata, color=['CD3D', 'CD3E','CD3G','MALAT1'], vmin=0)

In [32]:
# 1. 记录删除前的数量
old_n = adata.n_obs

# 2. 仅保留标签不等于 'cell' 的细胞
adata = adata[adata.obs['cell_subtype'] != 'cell', :].copy()

# 3. 清理掉 'cell' 这个已经没有细胞的类别名（防止在图例里留下一个空标签）
if hasattr(adata.obs['cell_subtype'], 'cat'):
    adata.obs['cell_subtype'] = adata.obs['cell_subtype'].cat.remove_unused_categories()

# 4. 打印结果确认
print(f"清理完成！")
print(f"删除了 {old_n - adata.n_obs} 个散乱细胞。")
print(f"当前剩余细胞总数: {adata.n_obs}")

In [33]:
sc.pl.umap(adata, color=['cell_subtype'], vmin=0)

In [45]:
adata_pDC = adata[adata.obs['cell_subtype']=='pDC'].copy()

In [46]:
adata_pDC.write("adata_pDC_final.h5ad")

In [41]:
# 1. 确保主表包含所有必要的类别名（包含子集里新加的 T_doublet）
# 获取子集里的所有类别
subset_categories = adata_pDC.obs['cell_subtype'].cat.categories.tolist()

for cat in subset_categories:
    if cat not in adata.obs['cell_subtype'].cat.categories:
        adata.obs['cell_subtype'] = adata.obs['cell_subtype'].cat.add_categories([cat])

# 2. 【核心修复】将子集标签转为字符串后再映射
# 这样避开了 Categorical 对象之间 strict dtype 的检查
adata.obs.loc[adata_pDC.obs_names, 'cell_subtype'] = adata_pDC.obs['cell_subtype'].astype(str)

print("✅ 同步成功！")

# 3. 验证
print("\n验证主表中 T_doublet 的数量：")
print(adata.obs['cell_subtype'].value_counts().reindex(['T_doublet']))

In [42]:
sc.pl.umap(adata, color=['cell_subtype'], vmin=0)

In [43]:
# 1. 记录剔除前的数量
old_n = adata.n_obs

# 2. 剔除标记为 'T_doublet' 的细胞
# 使用 ~ 表示“不等于”
adata = adata[adata.obs['cell_subtype'] != 'T_doublet', :].copy()

# 3. 清理已删除的类别标签（防止绘图图例中出现多余的空标签）
if hasattr(adata.obs['cell_subtype'], 'cat'):
    adata.obs['cell_subtype'] = adata.obs['cell_subtype'].cat.remove_unused_categories()

# 4. 统计结果
print(f"成功剔除 T_doublet！")
print(f"剔除数量: {old_n - adata.n_obs} 个细胞")
print(f"剩余总细胞数: {adata.n_obs}")

# 5. 可选：重置颜色映射（因为类别发生了变动）
if 'cell_subtype_colors' in adata.uns:
    del adata.uns['cell_subtype_colors']

In [44]:
sc.pl.umap(adata, color=['cell_subtype'], vmin=0)

In [68]:
import scanpy as sc
import pandas as pd
import matplotlib.pyplot as plt

short_name_map = {
'CD4 Naïve': 'CD4 Naïve', 'CD4 Memory': 'CD4 Memory', 'CD4 CTL': 'CD4 CTL', 'CD4 Treg': 'Treg',
'CD8 Naïve': 'CD8 Naïve', 'CD8 Memory': 'CD8 Memory', 'CD8 CTL': 'CD8 CTL',
'MAIT': 'MAIT', 'γδT': 'γδT', 'NKT': 'NKT',
'Naive B cell': 'Naive B', 'Memory B cell': 'Memory B', 'Transitional B cell': 'Transitional B', 'Plasma cell': 'Plasma cell',
'Classical Monocytes': 'cMonocytes', 'Non-classical Monocytes': 'ncMonocytes', 'Neutrophils': 'Neutrophils',
'cDC2': 'cDC2', 'pDC': 'pDC', 'ASDC': 'ASDC',
'Tissue-Resident Macrophages': 'trMac', 'Complement-Activated Macrophages': 'caMac',
'IFN-γ-Responsive Macrophages': 'ifnMac', 'Inflammatory Activated Macrophages': 'infMac',
'Phagocytic Inflammatory Macrophages': 'phMac', 'Metabolically Active Resting Macrophages': 'maMac',
'ILC2': 'ILC2', 'NK bright': 'NK bright', 'NK dim': 'NK dim', 'Transit. NK': 'Transit. NK'
}



In [69]:
# 1. 建立映射并添加新列
# .map() 会根据字典将原始名称替换为缩写
adata.obs['cell_subtype_short'] = adata.obs['cell_subtype'].map(short_name_map)

# 2. 检查是否有未匹配到的名称（可选）
unmapped = adata.obs[adata.obs['cell_subtype_short'].isna()]['cell_subtype'].unique()
if len(unmapped) > 0:
    print(f"提示：以下类别未在缩写字典中找到，已设为 NaN: {unmapped}")

# 3. 转换为分类对象 (Category) 并按字典给出的顺序排序
# 这样在绘图时，图例（Legend）会按照你字典定义的 T->B->M->ILC 顺序排列
ordered_categories = [name for name in short_name_map.values() if name in adata.obs['cell_subtype_short'].unique()]
adata.obs['cell_subtype_short'] = pd.Categorical(
    adata.obs['cell_subtype_short'], 
    categories=ordered_categories, 
    ordered=True
)

print("✅ 'cell_subtype_short' 列已成功添加到 adata.obs")

In [70]:
import scanpy as sc
import matplotlib.pyplot as plt
from adjustText import adjust_text

# 1. 绘图：设置较大尺寸，并开启 outline 增加可读性
fig, ax = plt.subplots(figsize=(10, 10))

sc.pl.umap(
    adata,
    color='cell_subtype_short',
    palette='tab20',
    size=10,             # 稍微调大点，方便 adjust_text 识别“点”的位置
    ax=ax,
    show=False,
    frameon=False,
    legend_loc='on data',
    legend_fontsize=10,
    legend_fontweight='bold',
    legend_fontoutline=3 # 加厚白色描边，文字更清晰
)

# 2. 获取所有的文字对象
texts = [t for t in ax.texts if t.get_text() != '']

# 3. 自动调整文字位置
if texts:
    # 这一步是为了告诉 adjust_text 不要把字盖在数据点上
    # 我们从 ax 中获取所有的散点路径集
    # 如果是普通的 sc.pl.umap，它通常是 ax.collections 中的第一个
    scatters = [coll for coll in ax.collections] 

    adjust_text(
        texts, 
        add_objects=scatters,      # 把细胞点也作为“排斥物”，防止字盖住点
        arrowprops=dict(arrowstyle='-', color='gray', lw=0.5, alpha=0.6), # 连线设为淡灰色
        expand_points=(1.5, 1.5),  # 增加点周围的排斥范围
        expand_text=(1.2, 1.5),    # 增加文字周围的排斥范围
        force_points=0.1,          # 文字离开点的力度
        force_text=0.2,            # 文字之间互相推开的力度
        ax=ax
    )

# 移除标题防止遮挡顶部文字
ax.set_title('')
plt.tight_layout()
plt.show()

In [6]:
import scanpy as sc
import matplotlib.pyplot as plt
from adjustText import adjust_text

# --- 1. 定义 30 种不重复的高对比度颜色 ---
# 分组逻辑：T细胞(蓝/绿系), B细胞(红/粉系), 髓系(灰/褐系), Mac(紫/黄系), NK(青/淡系)
my_palette = [
    # T Cell Lineage (10 colors)
    '#1f77b4', '#3182bd', '#6baed6', '#fdb462', # CD4 & Treg
    '#2ca02c', '#74c476', '#a1d99b',           # CD8
    '#008080', '#1d91c0', '#225ea8',           # MAIT, gdT, NKT
    # B Cell Lineage (4 colors)
    '#e31a1c', '#fb9a99', '#fdbf6f', '#911eb4',
    # Myeloid & DCs (6 colors)
    '#808080', '#545454', '#8b4513', '#ffed6f', '#bebada', '#bc8f8f',
    # Macrophages (6 colors) - 重点区分颜色
    '#e78ac3', '#8da0cb', '#a6d854', '#ffd92f', '#66c2a5', '#e5c494',
    # ILC / NK (4 colors)
    '#17becf', '#9edae5', '#00ced1', '#556b2f'
]

# --- 2. 应用颜色到 adata ---
# 获取当前排序好的分类
ordered_cats = adata.obs['cell_subtype_short'].cat.categories
# 将颜色列表赋值给 adata.uns 中的颜色插槽 (变量名必须为 {col}_colors)
adata.uns['cell_subtype_short_colors'] = my_palette[:len(ordered_cats)]

# --- 3. 绘图 ---
fig, ax = plt.subplots(figsize=(12, 12))

sc.pl.umap(
    adata,
    color='cell_subtype_short',
    size=6,             
    ax=ax,
    show=False,
    frameon=False,
    legend_loc='on data',
    legend_fontsize=9,
    legend_fontweight='bold',
    legend_fontoutline=3 # 加厚文字描边，防止在浅色背景上看不清
)

# --- 4. 优化标签排布 (adjust_text) ---
texts = [t for t in ax.texts if t.get_text() != '']
if texts:
    # 获取所有的散点，增加避让阻力
    scatters = [coll for coll in ax.collections if hasattr(coll, 'get_offsets')]
    
    adjust_text(
        texts, 
        add_objects=scatters, 
        # 增加连线，帮助识别极度重叠的区域（如巨噬细胞）
        arrowprops=dict(arrowstyle='-', color='black', lw=0.4, alpha=0.5),
        expand_points=(1.5, 1.8),  # 文本离开点的距离
        expand_text=(1.2, 1.4),    # 文本之间的互斥力度
        force_points=0.05,
        ax=ax
    )

ax.set_title("UMAP by Cell Subtype (Color Optimized)", fontsize=16)
plt.tight_layout()
plt.show()

In [7]:
import scanpy as sc
import pandas as pd
import matplotlib.pyplot as plt
from adjustText import adjust_text

# 精心挑选的莫兰迪配色方案
soft_color_map = {
    # --- T 细胞系 (清冷色调：牛仔蓝、灰绿、丁香紫) ---
    'CD4 Naïve': '#7294D4',       # 灰蓝色
    'CD4 Memory': '#A3B9D9',      # 浅钢蓝 (区分 Naïve，但不刺眼)
    'CD4 CTL': '#4B6DA1',         # 深牛仔蓝
    'Treg': '#E29E85',            # 砖红粉
    'CD8 Naïve': '#809B8E',       # 豆沙绿
    'CD8 Memory': '#C2D6C3',      # 浅艾草色
    'CD8 CTL': '#5E7D63',         # 深森绿
    'MAIT': '#76B7B2',            # 灰松石色
    'γδT': '#D4A5A5',             # 烟粉色
    'NKT': '#94839B',             # 灰紫色
    
    # --- B 细胞系 (暖色调：落日红、灰陶色、淡灰粉) ---
    'Naive B': '#D66B6B',         # 暖红色
    'Memory B': '#F2B8B8',        # 晚霞粉
    'Transitional B': '#A17D94',  # 枯玫瑰色
    'Plasma cell': '#C9B6D3',     # 薰衣草灰
    
    # --- 髓系 & DC (大地色系：矿石灰、陶土、黄泥) ---
    'cMonocytes': '#697A8C',      # 板岩灰
    'ncMonocytes': '#9CAAB8',     # 浅岩灰
    'Neutrophils': '#B6B59D',     # 草木灰
    'cDC2': '#D8C385',            # 哑光金
    'pDC': '#8E91B7',             # 朦胧紫
    'ASDC': '#D2B4DE',            # 淡紫色
    
    # --- 巨噬细胞群 (柔和高对比度) ---
    'trMac': '#E6AFB9',           # 浅山茶粉
    'caMac': '#E8CC8D',           # 浅土黄
    'ifnMac': '#A1D6E2',          # 冰蓝色
    'infMac': '#B98D7B',          # 咖啡棕
    'phMac': '#88918D',           # 冷灰
    'maMac': '#B3A194',           # 奶咖色

    # --- NK / ILC (轻快透明色：天青、雾蓝) ---
    'NK bright': '#A8D8EA',       # 云蓝色
    'NK dim': '#6892B0',          # 雾霾蓝
    'Transit. NK': '#C5A7D1',     # 淡丁香
    'ILC2': '#E5E4A3'             # 浅橄榄黄
}

# 应用颜色
obs_cats = adata.obs['cell_subtype_short'].cat.categories
colors_list = [soft_color_map[cat] for cat in obs_cats]
adata.uns['cell_subtype_short_colors'] = colors_list

In [8]:
# 3. 开始绘图
fig, ax = plt.subplots(figsize=(12, 12))

sc.pl.umap(
    adata,
    color='cell_subtype_short',
    size=6,             
    ax=ax,
    show=False,
    frameon=False,
    legend_loc='on data',
    legend_fontsize=10,
    legend_fontweight='bold',
    legend_fontoutline=3   # 文字白色描边，让文字在杂乱背景中跳出来
)

# 4. 优化标签排布 (获取所有文字对象)
texts = [t for t in ax.texts if t.get_text() != '']
if texts:
    # scatters 获取点的位置，帮助 adjust_text 排斥
    scatters = [coll for coll in ax.collections if hasattr(coll, 'get_offsets')]
    
    adjust_text(
        texts, 
        add_objects=scatters, 
        arrowprops=dict(arrowstyle='-', color='black', lw=0.4, alpha=0.5),
        expand_points=(1.5, 1.8),  
        expand_text=(1.2, 1.4),    
        force_points=0.05,
        ax=ax
    )

ax.set_title("UMAP by Cell Subtype (Fully Customized Palette)", fontsize=16)
plt.tight_layout()
plt.show()

In [9]:
import scanpy as sc
import pandas as pd
import matplotlib.pyplot as plt
from adjustText import adjust_text

# 1. 定义颜色映射 (保持不变)
soft_color_map = {
    'CD4 Naïve': '#7294D4', 'CD4 Memory': '#A3B9D9', 'CD4 CTL': '#4B6DA1', 'Treg': '#E29E85',
    'CD8 Naïve': '#809B8E', 'CD8 Memory': '#C2D6C3', 'CD8 CTL': '#5E7D63', 'MAIT': '#76B7B2',
    'γδT': '#D4A5A5', 'NKT': '#94839B', 'Naive B': '#D66B6B', 'Memory B': '#F2B8B8',
    'Transitional B': '#A17D94', 'Plasma cell': '#C9B6D3', 'cMonocytes': '#697A8C',
    'ncMonocytes': '#9CAAB8', 'Neutrophils': '#B6B59D', 'cDC2': '#D8C385', 'pDC': '#8E91B7',
    'ASDC': '#D2B4DE', 'trMac': '#E6AFB9', 'caMac': '#E8CC8D', 'ifnMac': '#A1D6E2',
    'infMac': '#B98D7B', 'phMac': '#88918D', 'maMac': '#B3A194', 'NK bright': '#A8D8EA',
    'NK dim': '#6892B0', 'Transit. NK': '#C5A7D1', 'ILC2': '#E5E4A3'
}

# 2. 统一设置颜色（确保两张图的色调一致）
# 即使某张图缺少某个细胞系，颜色索引也不会乱
if 'cell_subtype_short' in adata.obs.columns:
    # 确保类别顺序一致
    obs_cats = adata.obs['cell_subtype_short'].cat.categories
    colors_list = [soft_color_map.get(cat, '#CCCCCC') for cat in obs_cats]
    adata.uns['cell_subtype_short_colors'] = colors_list

# 3. 创建子图布局 (1行2列)
fig, axes = plt.subplots(1, 2, figsize=(20, 10))
tissues = ['PBMC', 'CSF']

for i, tissue in enumerate(tissues):
    ax = axes[i]
    # 切片数据
    adata_sub = adata[adata.obs['tissue'] == tissue].copy()
    
    # 绘制 UMAP
    sc.pl.umap(
        adata_sub,
        color='cell_subtype_short',
        size=8,
        ax=ax,
        show=False,
        frameon=False,
        legend_loc='on data',
        legend_fontsize=9,
        legend_fontweight='bold',
        legend_fontoutline=2
    )
    
    # 优化该子图的标签
    texts = [t for t in ax.texts if t.get_text() != '']
    if texts:
        scatters = [coll for coll in ax.collections if hasattr(coll, 'get_offsets')]
        adjust_text(
            texts, 
            add_objects=scatters, 
            arrowprops=dict(arrowstyle='-', color='black', lw=0.4, alpha=0.5),
            expand_points=(1.5, 1.8),  
            expand_text=(1.2, 1.4),    
            force_points=0.05,
            ax=ax
        )
    
    ax.set_title(f"Tissue: {tissue}", fontsize=18, fontweight='bold')

plt.tight_layout()
plt.show()

In [10]:
adata.write_h5ad("adata_final.h5ad")

In [ ]:
dir()

In [5]:
adata = sc.read_h5ad("adata_final.h5ad")

In [6]:
sc.pl.umap(adata,color = 'dataset')

In [14]:
adata